<a href="https://colab.research.google.com/github/BhumiPatil1507/YM_functionable_code/blob/main/Ym_fully_functional_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install RDKit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.8/31.8 MB 67.0 MB/s  0:00:00


In [ ]:

from google.colab import drive
drive.mount('/content/drive')

import os
import gc
import csv
import json
import numpy as np
import pandas as pd

# ============================================================
# PROJECT PATHS
# ============================================================

PROJECT_DIR = "/content/drive/MyDrive/Colab"

DATA_FILE = os.path.join(
    PROJECT_DIR, "polyOne_aa.csv"
)

FEATURE_DIR = os.path.join(
    PROJECT_DIR, "polymer_features"
)

OUTPUT_DIR = os.path.join(
    PROJECT_DIR, "YM_MOE_results"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

TARGET = "YM"
TEST_SIZE = 0.20
RANDOM_STATE = 42

FEATURE_FILES = {
    "fingerprints": os.path.join(
        FEATURE_DIR, "polyOne_aa_fingerprints.csv"
    ),
    "descriptors": os.path.join(
        FEATURE_DIR, "polyOne_aa_descriptors.csv"
    ),
    "graph": os.path.join(
        FEATURE_DIR, "polyOne_aa_graph.csv"
    ),
    "functional_groups": os.path.join(
        FEATURE_DIR, "polyOne_aa_funcgroups.csv"
    ),
}

print("Google Drive mounted.")
print("Original CSV:", DATA_FILE)
print("Feature directory:", FEATURE_DIR)
print("Output directory:", OUTPUT_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted.
Original CSV: /content/drive/MyDrive/Colab/polyOne_aa.csv
Feature directory: /content/drive/MyDrive/Colab/polymer_features
Output directory: /content/drive/MyDrive/Colab/YM_MOE_results


In [ ]:
assert os.path.exists(DATA_FILE), (
    f"Original CSV not found: {DATA_FILE}"
)

assert os.path.isdir(FEATURE_DIR), (
    f"Feature directory not found: {FEATURE_DIR}"
)

print("Original CSV: FOUND")

for name, path in FEATURE_FILES.items():
    status = "FOUND" if os.path.isfile(path) else "MISSING"
    print(f"{name:20s}: {status}")

print("\nOutput directory ready:", os.path.isdir(OUTPUT_DIR))

Original CSV: FOUND
fingerprints        : FOUND
descriptors         : FOUND
graph               : FOUND
functional_groups   : FOUND

Output directory ready: True


In [ ]:
def inspect_csv(path):
    with open(
        path, "r", encoding="utf-8-sig", newline=""
    ) as f:
        reader = csv.reader(f)
        header = next(reader, None)

    if header is None:
        raise ValueError(f"Empty CSV: {path}")

    with open(
        path, "r", encoding="utf-8-sig", newline=""
    ) as f:
        row_count = sum(1 for _ in f) - 1

    return header, row_count


for name, path in {
    "original": DATA_FILE,
    **FEATURE_FILES
}.items():

    if not os.path.exists(path):
        continue

    header, nrows = inspect_csv(path)

    print("\n" + "=" * 65)
    print("FILE:", name)
    print("ROWS:", nrows)
    print("COLUMNS:", len(header))
    print("FIRST 12 COLUMN NAMES:", header[:12])

    if name == "original":
        print("YM column exists:", TARGET in header)


FILE: original
ROWS: 500000
COLUMNS: 38
FIRST 12 COLUMN NAMES: ['smiles', 'Egc', 'Egb', 'Eib', 'CED', 'Ei', 'Eea', 'nc', 'ne', 'epse_6.0', 'epsc', 'epse_3.0']
YM column exists: True

FILE: fingerprints
ROWS: 500000
COLUMNS: 2086
FIRST 12 COLUMN NAMES: ['smiles', 'Egc', 'Egb', 'Eib', 'CED', 'Ei', 'Eea', 'nc', 'ne', 'epse_6.0', 'epsc', 'epse_3.0']

FILE: descriptors
ROWS: 500000
COLUMNS: 54
FIRST 12 COLUMN NAMES: ['smiles', 'Egc', 'Egb', 'Eib', 'CED', 'Ei', 'Eea', 'nc', 'ne', 'epse_6.0', 'epsc', 'epse_3.0']

FILE: graph
ROWS: 500000
COLUMNS: 55
FIRST 12 COLUMN NAMES: ['smiles', 'Egc', 'Egb', 'Eib', 'CED', 'Ei', 'Eea', 'nc', 'ne', 'epse_6.0', 'epsc', 'epse_3.0']

FILE: functional_groups
ROWS: 500000
COLUMNS: 71
FIRST 12 COLUMN NAMES: ['smiles', 'Egc', 'Egb', 'Eib', 'CED', 'Ei', 'Eea', 'nc', 'ne', 'epse_6.0', 'epsc', 'epse_3.0']


In [ ]:

# ============================================================
# CELL 7: LOAD BASE COLUMNS AND CHECK FEATURE FILES
# ============================================================

import os
import csv
import pandas as pd

# Read only the base CSV header; do not load 500,000 rows
with open(DATA_FILE, "r", encoding="utf-8-sig", newline="") as f:
    reader = csv.reader(f)
    BASE_COLUMNS = next(reader)

BASE_COLUMN_SET = set(BASE_COLUMNS)

print("Base CSV columns:", len(BASE_COLUMNS))
print("YM exists in base CSV:", TARGET in BASE_COLUMN_SET)

if TARGET not in BASE_COLUMN_SET:
    raise ValueError("YM target column is missing from the base CSV.")

# Columns that must not be used as model inputs
FORBIDDEN_COLUMNS = {
    "YM",
    "smiles",
    "target",
    "actual_ym",
    "predicted_ym"
}

EXPERT_FEATURE_COLUMNS = {}

for name, path in FEATURE_FILES.items():

    if not os.path.exists(path):
        raise FileNotFoundError(path)

    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        header = next(csv.reader(f))

    if TARGET not in header:
        raise ValueError(f"YM is missing from {name} CSV.")

    # Identify columns added by feature extraction
    extracted_columns = [
        col for col in header
        if col not in BASE_COLUMN_SET
        and col not in FORBIDDEN_COLUMNS
    ]

    EXPERT_FEATURE_COLUMNS[name] = extracted_columns

    print("\n" + "=" * 55)
    print("Expert:", name)
    print("Total CSV columns:", len(header))
    print("Additional extracted columns:", len(extracted_columns))
    print("Example extracted columns:", extracted_columns[:10])

print("\nFeature inspection completed successfully.")


Base CSV columns: 38
YM exists in base CSV: True

Expert: fingerprints
Total CSV columns: 2086
Additional extracted columns: 2048
Example extracted columns: ['fp_0', 'fp_1', 'fp_2', 'fp_3', 'fp_4', 'fp_5', 'fp_6', 'fp_7', 'fp_8', 'fp_9']

Expert: descriptors
Total CSV columns: 54
Additional extracted columns: 16
Example extracted columns: ['MolWt', 'HeavyAtomCount', 'NumHAcceptors', 'NumHDonors', 'MolLogP', 'TPSA', 'NumRotatableBonds', 'RingCount', 'NumAromaticRings', 'NumAliphaticRings']

Expert: graph
Total CSV columns: 55
Additional extracted columns: 17
Example extracted columns: ['graph_num_atoms', 'graph_num_bonds', 'graph_num_heavy_atoms', 'graph_num_wildcards', 'graph_mean_degree', 'graph_max_degree', 'graph_min_degree', 'graph_degree_std', 'graph_num_components', 'graph_num_rings']

Expert: functional_groups
Total CSV columns: 71
Additional extracted columns: 33
Example extracted columns: ['fg_alcohol', 'fg_phenol', 'fg_ether', 'fg_aldehyde', 'fg_ketone', 'fg_carboxylic_acid',

In [ ]:
# ============================================================
# CELL 9: TRAIN INDIVIDUAL YM EXPERTS
# ============================================================

%pip -q install xgboost scipy scikit-learn joblib

In [ ]:
from sklearn.model_selection import train_test_split

y_all = pd.to_numeric(
    pd.read_csv(DATA_FILE, usecols=[TARGET])[TARGET],
    errors="coerce"
)

valid_rows = np.flatnonzero(y_all.notna().to_numpy())
y = y_all.iloc[valid_rows].to_numpy(dtype=np.float32)

all_valid_indices = np.arange(len(y))

train_idx, test_idx = train_test_split(
    all_valid_indices,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    shuffle=True
)

y_train = y[train_idx]
y_test = y[test_idx]

np.save(os.path.join(OUTPUT_DIR, "YM_valid_rows.npy"), valid_rows)
np.save(os.path.join(OUTPUT_DIR, "YM_train_indices.npy"), train_idx)
np.save(os.path.join(OUTPUT_DIR, "YM_test_indices.npy"), test_idx)

print("Total original rows:", len(y_all))
print("Rows with valid YM:", len(y))
print("Training rows:", len(train_idx))
print("Test rows:", len(test_idx))
print("Training proportion:", round(len(train_idx) / len(y), 4))
print("Test proportion:", round(len(test_idx) / len(y), 4))
print("YM target range:", float(np.min(y)), "to", float(np.max(y)))

Total original rows: 500000
Rows with valid YM: 500000
Training rows: 400000
Test rows: 100000
Training proportion: 0.8
Test proportion: 0.2
YM target range: 183.67999267578125 to 3774.35498046875


In [ ]:
import os
import gc
import csv
import time
import json
import warnings
import numpy as np
import pandas as pd

from scipy.sparse import csr_matrix, vstack, issparse

warnings.filterwarnings("ignore")

CHUNK_SIZE = 25000
MAX_TREES = 2000
EARLY_STOPPING_ROUNDS = 100
N_OOF_FOLDS = 3

MODEL_DIR = os.path.join(OUTPUT_DIR, "models")
os.makedirs(MODEL_DIR, exist_ok=True)

# Confirm that the required variables are already defined
required_variables = [
    "DATA_FILE", "FEATURE_FILES", "EXPERT_FEATURE_COLUMNS",
    "OUTPUT_DIR", "valid_rows", "y", "train_idx", "test_idx",
    "y_train", "y_test"
]

missing = [v for v in required_variables if v not in globals()]
if missing:
    raise NameError(f"Run the setup/split cells first. Missing: {missing}")

# Verify that feature files contain the expected number of original rows
base_rows = len(y_all)

for name, path in FEATURE_FILES.items():
    if not os.path.isfile(path):
        raise FileNotFoundError(f"Feature file not found: {path}")

    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        header = next(csv.reader(f))

    missing_features = [
        col for col in EXPERT_FEATURE_COLUMNS[name]
        if col not in header
    ]
    if missing_features:
        raise ValueError(
            f"{name}: feature columns missing from file: "
            f"{missing_features[:5]}"
        )

print("Target: YM")
print("Training samples:", len(y_train))
print("Test samples:", len(y_test))
print("Maximum trees per expert:", MAX_TREES)
print("Feature files verified.")

Target: YM
Training samples: 400000
Test samples: 100000
Maximum trees per expert: 2000
Feature files verified.


In [ ]:

print("CELL 2 STARTED", flush=True)

import os
import csv
import time

print("\n1. Checking required variables...", flush=True)

for var in [
    "DATA_FILE",
    "FEATURE_FILES",
    "EXPERT_FEATURE_COLUMNS",
    "OUTPUT_DIR",
    "valid_rows",
    "y",
    "y_all"
]:
    print(f"{var}: {var in globals()}", flush=True)

print("\n2. Checking base CSV...", flush=True)

if "DATA_FILE" in globals():
    print("Path:", DATA_FILE, flush=True)
    print("Exists:", os.path.isfile(DATA_FILE), flush=True)

print("\n3. Checking feature files...", flush=True)

if "FEATURE_FILES" in globals():
    for name, path in FEATURE_FILES.items():
        print(
            name,
            "| Exists:", os.path.isfile(path),
            "| Path:", path,
            flush=True
        )

print("\n4. Checking extracted feature columns...", flush=True)

if "EXPERT_FEATURE_COLUMNS" in globals():
    for name, columns in EXPERT_FEATURE_COLUMNS.items():
        print(
            name,
            "| Feature count:", len(columns),
            "| First features:", columns[:5],
            flush=True
        )

print("\nCELL 2 DIAGNOSTIC FINISHED", flush=True)


CELL 2 STARTED

1. Checking required variables...
DATA_FILE: True
FEATURE_FILES: True
EXPERT_FEATURE_COLUMNS: True
OUTPUT_DIR: True
valid_rows: True
y: True
y_all: True

2. Checking base CSV...
Path: /content/drive/MyDrive/Colab/polyOne_aa.csv
Exists: True

3. Checking feature files...
fingerprints | Exists: True | Path: /content/drive/MyDrive/Colab/polymer_features/polyOne_aa_fingerprints.csv
descriptors | Exists: True | Path: /content/drive/MyDrive/Colab/polymer_features/polyOne_aa_descriptors.csv
graph | Exists: True | Path: /content/drive/MyDrive/Colab/polymer_features/polyOne_aa_graph.csv
functional_groups | Exists: True | Path: /content/drive/MyDrive/Colab/polymer_features/polyOne_aa_funcgroups.csv

4. Checking extracted feature columns...
fingerprints | Feature count: 2048 | First features: ['fp_0', 'fp_1', 'fp_2', 'fp_3', 'fp_4']
descriptors | Feature count: 16 | First features: ['MolWt', 'HeavyAtomCount', 'NumHAcceptors', 'NumHDonors', 'MolLogP']
graph | Feature count: 17 | Fi

In [ ]:
# ============================================================
# NEXT CELL: FINAL PRE-TRAINING CHECK FOR YM MoE
# ============================================================

import os
import numpy as np
import xgboost as xgb
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

print("XGBoost version:", xgb.__version__)
print("YM target:", TARGET)
print("Training rows:", len(train_idx))
print("Test rows:", len(test_idx))
print("Maximum trees per expert:", MAX_TREES)

print("\nFeature dimensions:")
for name, columns in EXPERT_FEATURE_COLUMNS.items():
    print(f"{name:20s}: {len(columns)} features")

print("\nSplit files:")
for filename in [
    "YM_valid_rows.npy",
    "YM_train_indices.npy",
    "YM_test_indices.npy"
]:
    path = os.path.join(OUTPUT_DIR, filename)
    print(f"{filename}: {'FOUND' if os.path.isfile(path) else 'MISSING'}")

assert len(train_idx) == 400000
assert len(test_idx) == 100000
assert set(EXPERT_FEATURE_COLUMNS.keys()) == {
    "fingerprints",
    "descriptors",
    "graph",
    "functional_groups"
}

print("\nPRE-TRAINING CHECK PASSED")

XGBoost version: 3.4.1
YM target: YM
Training rows: 400000
Test rows: 100000
Maximum trees per expert: 2000

Feature dimensions:
fingerprints        : 2048 features
descriptors         : 16 features
graph               : 17 features
functional_groups   : 33 features

Split files:
YM_valid_rows.npy: FOUND
YM_train_indices.npy: FOUND
YM_test_indices.npy: FOUND

PRE-TRAINING CHECK PASSED


In [ ]:
print("YM target:", TARGET)
print("Training rows:", len(train_idx))
print("Test rows:", len(test_idx))

print("\nFeature definitions:")
for name, path in FEATURE_FILES.items():
    print(f"{name}: {path}")

print("\nEXPERT_FEATURE_COLUMNS exists:",
      "EXPERT_FEATURE_COLUMNS" in globals())

if "EXPERT_FEATURE_COLUMNS" in globals():
    for name, cols in EXPERT_FEATURE_COLUMNS.items():
        print(f"{name}: {len(cols)} features")

YM target: YM
Training rows: 400000
Test rows: 100000

Feature definitions:
fingerprints: /content/drive/MyDrive/Colab/polymer_features/polyOne_aa_fingerprints.csv
descriptors: /content/drive/MyDrive/Colab/polymer_features/polyOne_aa_descriptors.csv
graph: /content/drive/MyDrive/Colab/polymer_features/polyOne_aa_graph.csv
functional_groups: /content/drive/MyDrive/Colab/polymer_features/polyOne_aa_funcgroups.csv

EXPERT_FEATURE_COLUMNS exists: True
fingerprints: 2048 features
descriptors: 16 features
graph: 17 features
functional_groups: 33 features


In [ ]:

# ============================================================
# YM MoE — CELL 1: DEFINE FOUR EXPERTS FROM EXISTING FILES
# Does not rerun feature extraction or change the split.
# ============================================================

import os
import csv
import json
import time
import gc
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import KFold, train_test_split
from scipy.sparse import csr_matrix, vstack, issparse

# ------------------------------------------------------------
# 1. Verify the existing setup
# ------------------------------------------------------------

assert TARGET == "YM"
assert len(train_idx) == 400000
assert len(test_idx) == 100000
assert len(valid_rows) == len(y)

print("YM target confirmed.")
print("Training rows:", len(train_idx))
print("Test rows:", len(test_idx))

# ------------------------------------------------------------
# 2. Identify extracted features from the CSV headers
# ------------------------------------------------------------

with open(DATA_FILE, "r", encoding="utf-8-sig", newline="") as f:
    base_columns = next(csv.reader(f))

EXPERT_FEATURE_COLUMNS = {}

for expert, path in FEATURE_FILES.items():
    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        columns = next(csv.reader(f))

    # Features added during extraction, not original base columns.
    extracted = [c for c in columns if c not in base_columns]

    if not extracted:
        raise ValueError(f"No extracted features found for {expert}")

    EXPERT_FEATURE_COLUMNS[expert] = extracted

    print(
        f"{expert:20s} | "
        f"{len(extracted):4d} features | "
        f"{os.path.basename(path)}"
    )

# ------------------------------------------------------------
# 3. Save the configuration
# ------------------------------------------------------------

os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(
    os.path.join(OUTPUT_DIR, "YM_expert_feature_columns.json"), "w"
) as f:
    json.dump(EXPERT_FEATURE_COLUMNS, f, indent=2)

print("\nFeature definitions saved.")
print("XGBoost version:", xgb.__version__)
print("CELL 1 COMPLETE")


YM target confirmed.
Training rows: 400000
Test rows: 100000
fingerprints         | 2048 features | polyOne_aa_fingerprints.csv
descriptors          |   16 features | polyOne_aa_descriptors.csv
graph                |   17 features | polyOne_aa_graph.csv
functional_groups    |   33 features | polyOne_aa_funcgroups.csv

Feature definitions saved.
XGBoost version: 3.4.1
CELL 1 COMPLETE


In [ ]:

# ============================================================
# CELL 2: TRAIN FOUR INDIVIDUAL YM EXPERTS
# ============================================================

import os
import gc
import json
import time
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split

MODEL_DIR = os.path.join(OUTPUT_DIR, "models")
PRED_DIR = os.path.join(OUTPUT_DIR, "predictions")
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(PRED_DIR, exist_ok=True)

MAX_TREES = 2000
CHUNK_SIZE = 25000
VALIDATION_SIZE = 0.10

print("Starting YM expert training", flush=True)

def load_expert_data(expert_name):
    """Load existing extracted features and align them with the YM split."""
    path = FEATURE_FILES[expert_name]
    columns = EXPERT_FEATURE_COLUMNS[expert_name]

    chunks = []
    for chunk in pd.read_csv(
        path, usecols=columns, chunksize=CHUNK_SIZE, low_memory=False
    ):
        chunk = chunk.apply(pd.to_numeric, errors="coerce")
        chunk = chunk.replace([np.inf, -np.inf], np.nan)
        chunks.append(chunk.to_numpy(dtype=np.float32))

    X = np.vstack(chunks)
    del chunks
    gc.collect()

    # Align feature rows with the valid YM target rows.
    X = X[valid_rows]

    X_train = X[train_idx]
    X_test = X[test_idx]

    # Impute missing values using training-set medians only.
    medians = np.nanmedian(X_train, axis=0)
    medians = np.where(np.isfinite(medians), medians, 0.0)

    for matrix in (X_train, X_test):
        bad_rows, bad_cols = np.where(~np.isfinite(matrix))
        if len(bad_rows):
            matrix[bad_rows, bad_cols] = medians[bad_cols]

    return X_train, X_test, medians


expert_results = {}
expert_test_predictions = {}

for expert_name, feature_columns in EXPERT_FEATURE_COLUMNS.items():

    model_path = os.path.join(
        MODEL_DIR, f"YM_{expert_name}_expert.json"
    )
    prediction_path = os.path.join(
        PRED_DIR, f"YM_{expert_name}_test_predictions.npy"
    )
    metrics_path = os.path.join(
        MODEL_DIR, f"YM_{expert_name}_metrics.json"
    )
    medians_path = os.path.join(
        MODEL_DIR, f"YM_{expert_name}_medians.npy"
    )

    # Resume experts already saved completely.
    if all(os.path.exists(p) for p in [
        model_path, prediction_path, metrics_path, medians_path
    ]):
        print(f"\nLoading completed expert: {expert_name}", flush=True)

        model = xgb.XGBRegressor()
        model.load_model(model_path)

        with open(metrics_path) as f:
            metrics = json.load(f)

        expert_results[expert_name] = metrics
        expert_test_predictions[expert_name] = np.load(prediction_path)

        print(
            f"R²={metrics['R2']:.5f} | "
            f"RMSE={metrics['RMSE']:.5f} | "
            f"MAE={metrics['MAE']:.5f}",
            flush=True,
        )
        del model
        gc.collect()
        continue

    print(
        f"\nLoading {expert_name}: {len(feature_columns)} features",
        flush=True,
    )
    start = time.time()

    X_train, X_test, medians = load_expert_data(expert_name)

    inner_train, val_idx = train_test_split(
        np.arange(len(y_train)),
        test_size=VALIDATION_SIZE,
        random_state=RANDOM_STATE,
        shuffle=True,
    )

    model = xgb.XGBRegressor(
        objective="reg:squarederror",
        n_estimators=MAX_TREES,
        max_depth=8,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        tree_method="hist",
        eval_metric="rmse",
        early_stopping_rounds=100,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )

    print(f"Training {expert_name}...", flush=True)

    model.fit(
        X_train[inner_train],
        y_train[inner_train],
        eval_set=[(X_train[val_idx], y_train[val_idx])],
        verbose=100,
    )

    pred = model.predict(X_test)

    metrics = {
        "target": "YM",
        "expert": expert_name,
        "feature_count": int(len(feature_columns)),
        "R2": float(r2_score(y_test, pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_test, pred))),
        "MAE": float(mean_absolute_error(y_test, pred)),
        "best_iteration": int(model.best_iteration),
        "trees_used": int(model.best_iteration + 1),
        "training_minutes": float((time.time() - start) / 60),
    }

    model.save_model(model_path)
    np.save(prediction_path, pred)
    np.save(medians_path, medians)

    with open(metrics_path, "w") as f:
        json.dump(metrics, f, indent=2)

    expert_results[expert_name] = metrics
    expert_test_predictions[expert_name] = pred

    print("\nExpert results:", json.dumps(metrics, indent=2), flush=True)

    del X_train, X_test, medians, model
    gc.collect()

# Save comparison of all four individual experts.
results_df = pd.DataFrame(expert_results).T
results_file = os.path.join(OUTPUT_DIR, "YM_individual_expert_metrics.csv")
results_df.to_csv(results_file, index_label="expert")

print("\n" + "=" * 65)
print("FOUR EXPERTS FINISHED")
print("=" * 65)
print(results_df[["R2", "RMSE", "MAE", "trees_used"]])
print("\nMetrics saved to:", results_file)
print("Models saved to:", MODEL_DIR)
print("Test predictions saved to:", PRED_DIR)


Starting YM expert training

Loading completed expert: fingerprints
R²=0.79703 | RMSE=161.03221 | MAE=118.36597

Loading completed expert: descriptors
R²=0.56035 | RMSE=237.00372 | MAE=179.58916

Loading completed expert: graph
R²=0.44014 | RMSE=267.44919 | MAE=204.09711

Loading completed expert: functional_groups
R²=0.19591 | RMSE=320.52001 | MAE=238.64687

FOUR EXPERTS FINISHED
                         R2        RMSE         MAE trees_used
fingerprints       0.797035  161.032205  118.365967       2000
descriptors         0.56035  237.003725  179.589157       2000
graph              0.440141  267.449192  204.097107       1986
functional_groups  0.195906  320.520012  238.646866        561

Metrics saved to: /content/drive/MyDrive/Colab/YM_MOE_results/YM_individual_expert_metrics.csv
Models saved to: /content/drive/MyDrive/Colab/YM_MOE_results/models
Test predictions saved to: /content/drive/MyDrive/Colab/YM_MOE_results/predictions


In [ ]:
# ================================================================
# YM OOF GENERATION FROM SCRATCH
# 4 experts x 3 folds = 12 OOF models
# Original expert models and train/test split remain untouched.
# ================================================================

import os
import gc
import csv
import json
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.model_selection import KFold
from sklearn.impute import SimpleImputer

# -------------------- PATHS --------------------

PROJECT_DIR = "/content/drive/MyDrive/Colab"
DATA_FILE = os.path.join(PROJECT_DIR, "polyOne_aa.csv")
FEATURE_DIR = os.path.join(PROJECT_DIR, "polymer_features")

OUTPUT_DIR = os.path.join(PROJECT_DIR, "YM_MOE_results")
MODEL_DIR = os.path.join(OUTPUT_DIR, "models")
OOF_DIR = os.path.join(OUTPUT_DIR, "oof")

os.makedirs(OOF_DIR, exist_ok=True)

FEATURE_FILES = {
    "fingerprints": os.path.join(
        FEATURE_DIR, "polyOne_aa_fingerprints.csv"
    ),
    "descriptors": os.path.join(
        FEATURE_DIR, "polyOne_aa_descriptors.csv"
    ),
    "graph": os.path.join(
        FEATURE_DIR, "polyOne_aa_graph.csv"
    ),
    "functional_groups": os.path.join(
        FEATURE_DIR, "polyOne_aa_funcgroups.csv"
    ),
}

EXPERTS = [
    "fingerprints",
    "descriptors",
    "graph",
    "functional_groups",
]

TARGET = "YM"
N_FOLDS = 3
CHUNK_SIZE = 5000
SEED = 42

OOF_PATH = os.path.join(OOF_DIR, "YM_oof_predictions.npy")
PROGRESS_PATH = os.path.join(OOF_DIR, "YM_oof_progress.json")

# -------------------- CHECK INPUT FILES --------------------

for path in [
    DATA_FILE,
    *FEATURE_FILES.values(),
    os.path.join(OUTPUT_DIR, "YM_train_indices.npy"),
    os.path.join(OUTPUT_DIR, "YM_test_indices.npy"),
]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required file missing: {path}")

for expert in EXPERTS:
    model_path = os.path.join(
        MODEL_DIR, f"YM_{expert}_expert.json"
    )
    metrics_path = os.path.join(
        MODEL_DIR, f"YM_{expert}_metrics.json"
    )

    if not os.path.exists(model_path):
        raise FileNotFoundError(model_path)

    if not os.path.exists(metrics_path):
        raise FileNotFoundError(metrics_path)

print("All input files found.")

# -------------------- RESTORE EXISTING SPLIT --------------------

train_idx = np.load(
    os.path.join(OUTPUT_DIR, "YM_train_indices.npy")
).astype(np.int64)

test_idx = np.load(
    os.path.join(OUTPUT_DIR, "YM_test_indices.npy")
).astype(np.int64)

if np.intersect1d(train_idx, test_idx).size:
    raise ValueError("Train/test split overlaps!")

if len(train_idx) != 400_000 or len(test_idx) != 100_000:
    raise ValueError(
        f"Unexpected split sizes: train={len(train_idx)}, "
        f"test={len(test_idx)}. Stopping to protect the existing split."
    )

# Load target only.
y_df = pd.read_csv(DATA_FILE, usecols=[TARGET], low_memory=False)
y_all = pd.to_numeric(
    y_df[TARGET], errors="coerce"
).to_numpy(dtype=np.float32)
del y_df
gc.collect()

if not np.isfinite(y_all[train_idx]).all():
    raise ValueError("YM target has missing/non-finite training values.")

y_train = y_all[train_idx]

print(f"Training rows: {len(train_idx):,}")
print(f"Test rows:     {len(test_idx):,}")

# -------------------- IDENTIFY FEATURE COLUMNS --------------------

with open(DATA_FILE, "r", encoding="utf-8-sig", newline="") as f:
    base_columns = next(csv.reader(f))

FEATURE_COLUMNS = {}

for expert, path in FEATURE_FILES.items():
    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        columns = next(csv.reader(f))

    features = [c for c in columns if c not in base_columns]

    if not features:
        raise ValueError(f"No feature columns identified for {expert}.")

    FEATURE_COLUMNS[expert] = features
    print(f"{expert}: {len(features)} features")

# -------------------- MATCH TREE COUNTS TO SAVED EXPERTS --------------------
# Use each saved expert's actual boosting-round count.
# Do not retrain or overwrite the original expert models.

TREE_COUNTS = {}

for expert in EXPERTS:
    saved_model_path = os.path.join(
        MODEL_DIR, f"YM_{expert}_expert.json"
    )

    saved_model = xgb.XGBRegressor()
    saved_model.load_model(saved_model_path)

    TREE_COUNTS[expert] = int(
        saved_model.get_booster().num_boosted_rounds()
    )

    del saved_model

print("\nOOF boosting rounds:")
for expert, count in TREE_COUNTS.items():
    print(f"  {expert}: {count}")

# -------------------- MEMORY-AWARE ROW LOADER --------------------

def load_feature_rows(expert, requested_rows):
    """
    Load selected original CSV rows in chunks.
    The returned array follows requested_rows order.
    Fingerprints are checked for missing/non-finite values.
    """

    path = FEATURE_FILES[expert]
    columns = FEATURE_COLUMNS[expert]

    requested_rows = np.asarray(requested_rows, dtype=np.int64)

    # Sort for sequential CSV scanning; restore requested order later.
    order = np.argsort(requested_rows)
    sorted_rows = requested_rows[order]

    X = np.empty(
        (len(sorted_rows), len(columns)),
        dtype=np.float32
    )

    cursor = 0
    csv_start = 0

    for chunk in pd.read_csv(
        path,
        usecols=columns,
        chunksize=CHUNK_SIZE,
        low_memory=False
    ):
        csv_end = csv_start + len(chunk)

        while cursor < len(sorted_rows) and sorted_rows[cursor] < csv_end:
            cursor += 1

        # Find selected rows belonging to this chunk.
        previous_cursor = getattr(
            load_feature_rows, "_previous_cursor", None
        )

        # Use searchsorted for the current chunk boundaries.
        left = np.searchsorted(sorted_rows, csv_start, side="left")
        right = np.searchsorted(sorted_rows, csv_end, side="left")

        if right > left:
            local_idx = sorted_rows[left:right] - csv_start

            block = chunk.iloc[local_idx].to_numpy(
                dtype=np.float32,
                copy=True
            )

            X[left:right] = block
            del block

        csv_start = csv_end
        del chunk

        if csv_start > sorted_rows[-1] if len(sorted_rows) else True:
            break

    if len(sorted_rows) and csv_start <= sorted_rows[-1]:
        raise ValueError(
            f"Could not locate all requested rows in {path}."
        )

    if expert == "fingerprints":
        if not np.isfinite(X).all():
            raise ValueError(
                "Fingerprints contain missing/non-finite values. "
                "No imputation was applied."
            )

    # Restore the exact row order requested by the caller.
    inverse_order = np.empty_like(order)
    inverse_order[order] = np.arange(len(order))
    X = X[inverse_order]

    gc.collect()
    return X

# -------------------- CREATE A FRESH OOF FILE --------------------
# This intentionally starts OOF generation from scratch.

if os.path.exists(OOF_PATH):
    os.remove(OOF_PATH)

if os.path.exists(PROGRESS_PATH):
    os.remove(PROGRESS_PATH)

n_train = len(train_idx)
oof = np.lib.format.open_memmap(
    OOF_PATH,
    mode="w+",
    dtype=np.float32,
    shape=(n_train, len(EXPERTS))
)
oof[:] = np.nan
oof.flush()

progress = {
    "target": TARGET,
    "n_folds": N_FOLDS,
    "train_rows": int(n_train),
    "experts": EXPERTS,
    "completed": []
}

with open(PROGRESS_PATH, "w") as f:
    json.dump(progress, f, indent=2)

# -------------------- GENERATE OOF PREDICTIONS --------------------

kf = KFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=SEED
)

for fold, (fit_pos, holdout_pos) in enumerate(
    kf.split(np.arange(n_train)), start=1
):
    print(f"\n{'=' * 60}")
    print(f"FOLD {fold}/{N_FOLDS}")
    print(f"Fit rows:     {len(fit_pos):,}")
    print(f"Holdout rows: {len(holdout_pos):,}")
    print(f"{'=' * 60}")

    fit_rows = train_idx[fit_pos]
    holdout_rows = train_idx[holdout_pos]
    y_fit = y_train[fit_pos]

    for expert_idx, expert in enumerate(EXPERTS):
        print(f"\n[{fold}/{N_FOLDS}] Expert: {expert}")

        X_fit = load_feature_rows(expert, fit_rows)
        X_hold = load_feature_rows(expert, holdout_rows)

        # Fingerprints are used as-is.
        # Compact experts get fold-specific median imputation.
        if expert != "fingerprints":
            imputer = SimpleImputer(strategy="median")
            X_fit = imputer.fit_transform(X_fit).astype(
                np.float32, copy=False
            )
            X_hold = imputer.transform(X_hold).astype(
                np.float32, copy=False
            )

            if not np.isfinite(X_fit).all():
                raise ValueError(
                    f"Non-finite values remain for {expert}."
                )
            if not np.isfinite(X_hold).all():
                raise ValueError(
                    f"Non-finite holdout values remain for {expert}."
                )

        model = xgb.XGBRegressor(
            n_estimators=TREE_COUNTS[expert],
            tree_method="hist",
            objective="reg:squarederror",
            eval_metric="rmse",
            n_jobs=2,
            random_state=SEED + fold,
            verbosity=0
        )

        model.fit(X_fit, y_fit)

        predictions = model.predict(X_hold).astype(np.float32)

        oof[holdout_pos, expert_idx] = predictions
        oof.flush()

        progress["completed"].append(
            f"{expert}_fold_{fold}"
        )

        with open(PROGRESS_PATH, "w") as f:
            json.dump(progress, f, indent=2)

        print(
            f"Saved {len(predictions):,} OOF predictions "
            f"for {expert}, fold {fold}."
        )

        del X_fit, X_hold, model, predictions
        if expert != "fingerprints":
            del imputer
        gc.collect()

    del fit_rows, holdout_rows, y_fit
    gc.collect()

# -------------------- VALIDATE AND SAVE --------------------

oof.flush()

if not np.isfinite(oof).all():
    raise ValueError(
        "OOF matrix contains missing/non-finite predictions. "
        "Check the saved progress file before proceeding."
    )

print("\n" + "=" * 65)
print("OOF GENERATION SUCCESSFULLY COMPLETED")
print("=" * 65)
print("OOF file:", OOF_PATH)
print("Progress file:", PROGRESS_PATH)
print("OOF shape:", oof.shape)
print("Column order:", EXPERTS)
print("Completed tasks:", len(progress["completed"]), "/", N_FOLDS * len(EXPERTS))
print("All OOF predictions are finite.")

del oof
gc.collect()

All input files found.
Training rows: 400,000
Test rows:     100,000
fingerprints: 2048 features
descriptors: 16 features
graph: 17 features
functional_groups: 33 features

OOF boosting rounds:
  fingerprints: 2000
  descriptors: 2000
  graph: 2000
  functional_groups: 661

FOLD 1/3
Fit rows:     266,666
Holdout rows: 133,334

[1/3] Expert: fingerprints
Saved 133,334 OOF predictions for fingerprints, fold 1.

[1/3] Expert: descriptors
Saved 133,334 OOF predictions for descriptors, fold 1.

[1/3] Expert: graph
Saved 133,334 OOF predictions for graph, fold 1.

[1/3] Expert: functional_groups
Saved 133,334 OOF predictions for functional_groups, fold 1.

FOLD 2/3
Fit rows:     266,667
Holdout rows: 133,333

[2/3] Expert: fingerprints
Saved 133,333 OOF predictions for fingerprints, fold 2.

[2/3] Expert: descriptors
Saved 133,333 OOF predictions for descriptors, fold 2.

[2/3] Expert: graph
Saved 133,333 OOF predictions for graph, fold 2.

[2/3] Expert: functional_groups
Saved 133,333 OOF p

0

In [ ]:
# ================================================================
# YM MOE ROUTER — TRAIN, EVALUATE, AND CALCULATE DISAGREEMENT
# Reuses completed OOF predictions and existing test predictions.
# ================================================================

import os
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

# -------------------- PATHS --------------------

PROJECT_DIR = "/content/drive/MyDrive/Colab"
OUTPUT_DIR = os.path.join(PROJECT_DIR, "YM_MOE_results")
MODEL_DIR = os.path.join(OUTPUT_DIR, "models")
PRED_DIR = os.path.join(OUTPUT_DIR, "predictions")
OOF_DIR = os.path.join(OUTPUT_DIR, "oof")

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(PRED_DIR, exist_ok=True)

OOF_PATH = os.path.join(OOF_DIR, "YM_oof_predictions.npy")
DATA_FILE = os.path.join(PROJECT_DIR, "polyOne_aa.csv")

EXPERTS = [
    "fingerprints",
    "descriptors",
    "graph",
    "functional_groups",
]

# -------------------- LOAD AND VALIDATE --------------------

train_idx = np.load(
    os.path.join(OUTPUT_DIR, "YM_train_indices.npy")
)
test_idx = np.load(
    os.path.join(OUTPUT_DIR, "YM_test_indices.npy")
)

oof = np.load(OOF_PATH, mmap_mode="r")

if oof.shape != (len(train_idx), len(EXPERTS)):
    raise ValueError(f"Unexpected OOF shape: {oof.shape}")

if not np.isfinite(oof).all():
    raise ValueError("OOF predictions contain non-finite values.")

# Load actual YM values.
target_df = pd.read_csv(DATA_FILE, usecols=["YM"], low_memory=False)
y_all = pd.to_numeric(
    target_df["YM"], errors="coerce"
).to_numpy(dtype=np.float64)
del target_df

y_train = y_all[train_idx]
y_test = y_all[test_idx]

if not np.isfinite(y_train).all() or not np.isfinite(y_test).all():
    raise ValueError("YM target contains missing/non-finite values.")

# Load existing predictions from the four final expert models.
test_predictions = []

for expert in EXPERTS:
    path = os.path.join(
        PRED_DIR, f"YM_{expert}_test_predictions.npy"
    )

    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Missing saved test predictions: {path}"
        )

    pred = np.load(path).astype(np.float64)

    if len(pred) != len(test_idx):
        raise ValueError(
            f"{expert} predictions have {len(pred)} rows; "
            f"expected {len(test_idx)}."
        )

    if not np.isfinite(pred).all():
        raise ValueError(
            f"{expert} test predictions contain non-finite values."
        )

    test_predictions.append(pred)

test_predictions = np.column_stack(test_predictions)

print("OOF matrix:", oof.shape)
print("Test expert prediction matrix:", test_predictions.shape)
print("Train/test split reused successfully.")

# -------------------- TRAIN ROUTER --------------------

print("\nTraining ExtraTrees MoE router...")

router = ExtraTreesRegressor(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=20,
    max_features=1.0,
    n_jobs=-1,
    random_state=42
)

router.fit(np.asarray(oof), y_train)

ROUTER_PATH = os.path.join(
    MODEL_DIR, "YM_MoE_router.joblib"
)
joblib.dump(router, ROUTER_PATH)

print("Router saved:", ROUTER_PATH)

# -------------------- PREDICT TEST SET --------------------

moe_predictions = router.predict(test_predictions)

# Disagreement uncertainty:
# sample standard deviation across the four expert predictions.
D_i = np.std(test_predictions, axis=1, ddof=1)

absolute_error = np.abs(y_test - moe_predictions)

# -------------------- METRICS --------------------

def calculate_metrics(y_true, y_pred):
    return {
        "R2": r2_score(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred)
    }

rows = []

for i, expert in enumerate(EXPERTS):
    metrics = calculate_metrics(y_test, test_predictions[:, i])

    rows.append({
        "Model": expert,
        **metrics
    })

moe_metrics = calculate_metrics(y_test, moe_predictions)

rows.append({
    "Model": "MoE",
    **moe_metrics
})

metrics_df = pd.DataFrame(rows)

METRICS_PATH = os.path.join(
    OUTPUT_DIR, "YM_MoE_comparison_metrics.csv"
)
metrics_df.to_csv(METRICS_PATH, index=False)

# -------------------- UNCERTAINTY ANALYSIS --------------------

if np.std(D_i) > 0 and np.std(absolute_error) > 0:
    disagreement_error_corr = np.corrcoef(
        D_i, absolute_error
    )[0, 1]
else:
    disagreement_error_corr = np.nan

uncertainty_df = pd.DataFrame({
    "Actual_YM": y_test,
    "MoE_Predicted_YM": moe_predictions,
    "Absolute_Error": absolute_error,
    "Disagreement_Di": D_i,
})

for i, expert in enumerate(EXPERTS):
    uncertainty_df[f"{expert}_Prediction"] = test_predictions[:, i]

UNCERTAINTY_PATH = os.path.join(
    PRED_DIR, "YM_MoE_test_predictions_and_uncertainty.csv"
)
uncertainty_df.to_csv(UNCERTAINTY_PATH, index=False)

# -------------------- REPORT --------------------

print("\n" + "=" * 75)
print("YM EXPERTS vs MOE — UNTOUCHED TEST SET")
print("=" * 75)
print(metrics_df.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

best_expert_r2 = max(
    row["R2"] for row in rows if row["Model"] != "MoE"
)

print("\n" + "=" * 75)
print("DISAGREEMENT UNCERTAINTY")
print("=" * 75)
print(f"Mean Di: {np.mean(D_i):.6f}")
print(f"Median Di: {np.median(D_i):.6f}")
print(f"95th percentile Di: {np.percentile(D_i, 95):.6f}")
print(
    "Correlation between Di and absolute MoE error: "
    f"{disagreement_error_corr:.6f}"
)

print("\n" + "=" * 75)
print("MOE IMPROVEMENT")
print("=" * 75)
print(f"Best individual expert R²: {best_expert_r2:.6f}")
print(f"MoE R²:                    {moe_metrics['R2']:.6f}")
print(
    "MoE R² improvement:        "
    f"{moe_metrics['R2'] - best_expert_r2:+.6f}"
)

print("\nSaved outputs:")
print("Router:", ROUTER_PATH)
print("Metrics:", METRICS_PATH)
print("Test predictions + Di:", UNCERTAINTY_PATH)

OOF matrix: (400000, 4)
Test expert prediction matrix: (100000, 4)
Train/test split reused successfully.

Training ExtraTrees MoE router...
Router saved: /content/drive/MyDrive/Colab/YM_MOE_results/models/YM_MoE_router.joblib

YM EXPERTS vs MOE — UNTOUCHED TEST SET
            Model       R2       RMSE        MAE
     fingerprints 0.797035 161.032213 118.365963
      descriptors 0.560350 237.003725 179.589153
            graph 0.440141 267.449178 204.097090
functional_groups 0.195906 320.519996 238.646861
              MoE 0.809041 156.196747 115.758015

DISAGREEMENT UNCERTAINTY
Mean Di: 130.789469
Median Di: 111.279302
95th percentile Di: 288.693692
Correlation between Di and absolute MoE error: 0.218604

MOE IMPROVEMENT
Best individual expert R²: 0.797035
MoE R²:                    0.809041
MoE R² improvement:        +0.012006

Saved outputs:
Router: /content/drive/MyDrive/Colab/YM_MOE_results/models/YM_MoE_router.joblib
Metrics: /content/drive/MyDrive/Colab/YM_MOE_results/YM_MoE_com

In [ ]:
%%writefile /content/extract_features.py
"""
extract_features.py — chunked RDKit feature extraction compatible with batch_feat_ext.py.

This is a newly written implementation, not the unavailable original extractor.
It creates four aligned CSVs and retains source columns (including YM labels).

Outputs:
  {stem}_fingerprints.csv  : 2048 Morgan/ECFP4 bits (default)
  {stem}_descriptors.csv   : 16 RDKit descriptors
  {stem}_graph.csv         : 17 graph/topology features
  {stem}_funcgroups.csv    : 33 SMARTS indicators

The exact 16/17/33 definitions are newly specified here; they may differ from
the original senior implementation even though the dimensions match the README.
"""

from __future__ import annotations
import argparse
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import Descriptors, Lipinski, rdFingerprintGenerator, rdMolDescriptors

RDLogger.DisableLog("rdApp.error")
RDLogger.DisableLog("rdApp.warning")

DESCRIPTOR_FUNCS = {
    "MolWt": Descriptors.MolWt,
    "HeavyAtomCount": Lipinski.HeavyAtomCount,
    "NumHAcceptors": Lipinski.NumHAcceptors,
    "NumHDonors": Lipinski.NumHDonors,
    "MolLogP": Descriptors.MolLogP,
    "TPSA": rdMolDescriptors.CalcTPSA,
    "NumRotatableBonds": Lipinski.NumRotatableBonds,
    "RingCount": Lipinski.RingCount,
    "NumAromaticRings": Lipinski.NumAromaticRings,
    "NumAliphaticRings": Lipinski.NumAliphaticRings,
    "FractionCSP3": rdMolDescriptors.CalcFractionCSP3,
    "LabuteASA": rdMolDescriptors.CalcLabuteASA,
    "BalabanJ": Descriptors.BalabanJ,
    "BertzCT": Descriptors.BertzCT,
    "HallKierAlpha": Descriptors.HallKierAlpha,
    "Chi0v": Descriptors.Chi0v,
}
DESCRIPTOR_NAMES = list(DESCRIPTOR_FUNCS)

GRAPH_NAMES = [
    "graph_num_atoms", "graph_num_bonds", "graph_num_heavy_atoms",
    "graph_num_wildcards", "graph_mean_degree", "graph_max_degree",
    "graph_min_degree", "graph_degree_std", "graph_num_components",
    "graph_num_rings", "graph_num_aromatic_atoms", "graph_num_aromatic_bonds",
    "graph_num_single_bonds", "graph_num_double_bonds", "graph_num_triple_bonds",
    "graph_cycle_rank", "graph_fraction_heavy_atoms_in_rings",
]

# 33 functional-group/substructure indicators. These are explicit new definitions.
FUNCTIONAL_GROUP_SMARTS = [
    ("fg_alcohol", "[OX2H][CX4]"),
    ("fg_phenol", "[OX2H]c"),
    ("fg_ether", "[OD2]([#6])[#6]"),
    ("fg_aldehyde", "[CX3H1](=O)[#6]"),
    ("fg_ketone", "[#6][CX3](=O)[#6]"),
    ("fg_carboxylic_acid", "[CX3](=O)[OX2H1]"),
    ("fg_ester", "[CX3](=O)[OX2H0][#6]"),
    ("fg_amide", "[NX3][CX3](=[OX1])[#6]"),
    ("fg_primary_amine", "[NX3;H2;!$(NC=O)]"),
    ("fg_secondary_amine", "[NX3;H1;!$(NC=O)]([#6])[#6]"),
    ("fg_tertiary_amine", "[NX3;H0;!$(NC=O)]([#6])([#6])[#6]"),
    ("fg_imine", "[CX3]=[NX2]"),
    ("fg_nitrile", "[CX2]#N"),
    ("fg_nitro", "[N+](=O)[O-]"),
    ("fg_nitro_alt", "N(=O)=O"),
    ("fg_azo", "[NX2]=[NX2]"),
    ("fg_aromatic_n", "[n]"),
    ("fg_pyridine_like_n", "[nH0]"),
    ("fg_aromatic_oxygen", "[o]"),
    ("fg_aromatic_sulfur", "[s]"),
    ("fg_fluorine", "[F]"),
    ("fg_chlorine", "[Cl]"),
    ("fg_bromine", "[Br]"),
    ("fg_iodine", "[I]"),
    ("fg_alkene", "[CX3]=[CX3]"),
    ("fg_alkyne", "[CX2]#[CX2]"),
    ("fg_aromatic_ring", "a1aaaaa1"),
    ("fg_epoxide_like", "[OX2r3]1[CX4r3][CX4r3]1"),
    ("fg_silicon", "[Si]"),
    ("fg_phosphorus", "[P]"),
    ("fg_sulfonyl", "[SX4](=O)(=O)"),
    ("fg_sulfide", "[#16X2]([#6])[#6]"),
    ("fg_carbonate", "[OX2][CX3](=O)[OX2]"),
]
_COMPILED_FG = []
for _name, _smarts in FUNCTIONAL_GROUP_SMARTS:
    _query = Chem.MolFromSmarts(_smarts)
    if _query is None:
        raise ValueError(f"Invalid functional-group SMARTS: {_name}: {_smarts}")
    _COMPILED_FG.append((_name, _query))
FG_NAMES = [name for name, _ in FUNCTIONAL_GROUP_SMARTS]


def _find_smiles_column(columns):
    candidates = ("smiles", "SMILES", "canonical_smiles", "CanonicalSMILES")
    for candidate in candidates:
        if candidate in columns:
            return candidate
    by_lower = {str(c).strip().lower(): c for c in columns}
    for candidate in ("smiles", "canonical_smiles"):
        if candidate in by_lower:
            return by_lower[candidate]
    raise ValueError(f"No SMILES column found. Columns: {list(columns)}")


def _descriptor_values(mol):
    out = {}
    for name, fn in DESCRIPTOR_FUNCS.items():
        try:
            value = float(fn(mol))
            out[name] = value if np.isfinite(value) else np.nan
        except Exception:
            out[name] = np.nan
    return out


def _graph_values(mol):
    n_atoms, n_bonds, heavy = mol.GetNumAtoms(), mol.GetNumBonds(), mol.GetNumHeavyAtoms()
    degrees = np.asarray([a.GetDegree() for a in mol.GetAtoms()], dtype=float)
    if not degrees.size:
        degrees = np.array([0.0])
    components = len(Chem.GetMolFrags(mol))
    heavy_in_rings = sum(a.GetAtomicNum() > 1 and a.IsInRing() for a in mol.GetAtoms())
    return {
        "graph_num_atoms": float(n_atoms),
        "graph_num_bonds": float(n_bonds),
        "graph_num_heavy_atoms": float(heavy),
        "graph_num_wildcards": float(sum(a.GetAtomicNum() == 0 for a in mol.GetAtoms())),
        "graph_mean_degree": float(degrees.mean()),
        "graph_max_degree": float(degrees.max()),
        "graph_min_degree": float(degrees.min()),
        "graph_degree_std": float(degrees.std()),
        "graph_num_components": float(components),
        "graph_num_rings": float(rdMolDescriptors.CalcNumRings(mol)),
        "graph_num_aromatic_atoms": float(sum(a.GetIsAromatic() for a in mol.GetAtoms())),
        "graph_num_aromatic_bonds": float(sum(b.GetIsAromatic() for b in mol.GetBonds())),
        "graph_num_single_bonds": float(sum(b.GetBondType() == Chem.BondType.SINGLE for b in mol.GetBonds())),
        "graph_num_double_bonds": float(sum(b.GetBondType() == Chem.BondType.DOUBLE for b in mol.GetBonds())),
        "graph_num_triple_bonds": float(sum(b.GetBondType() == Chem.BondType.TRIPLE for b in mol.GetBonds())),
        "graph_cycle_rank": float(max(0, n_bonds - n_atoms + components)),
        "graph_fraction_heavy_atoms_in_rings": float(heavy_in_rings / heavy) if heavy else 0.0,
    }


def _fingerprint_values(mol, generator, nbits):
    fp = generator.GetFingerprint(mol)
    arr = np.zeros((nbits,), dtype=np.uint8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return {f"fp_{i}": int(arr[i]) for i in range(nbits)}


def process_csv_file(csv_path, output_dir, stem, morgan_radius=2,
                     morgan_nbits=2048, chunksize=1000):
    """Function signature and returned stats match batch_feat_ext.py."""
    csv_path, output_dir = Path(csv_path), Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    header = pd.read_csv(csv_path, nrows=0)
    smiles_col = _find_smiles_column(header.columns)
    # Avoid accidentally carrying old feature columns into regenerated outputs.
    base_cols = [
        c for c in header.columns
        if not (str(c).startswith(("fp_", "graph_", "fg_"))
                or str(c) in DESCRIPTOR_NAMES)
    ]
    output_paths = {
        ft: output_dir / f"{stem}_{ft}.csv"
        for ft in ("descriptors", "graph", "funcgroups", "fingerprints")
    }
    # New run: overwrite all four outputs. Process in chunks to avoid retaining
    # 500k x 2048 Python objects in RAM.
    for path in output_paths.values():
        if path.exists():
            path.unlink()

    generator = rdFingerprintGenerator.GetMorganGenerator(
        radius=int(morgan_radius), fpSize=int(morgan_nbits), includeChirality=False
    )
    total_rows = total_success = total_failed = 0
    start = time.time()

    for chunk in pd.read_csv(csv_path, chunksize=int(chunksize)):
        # Keep source columns and order; all four output files use the same rows.
        base = chunk[base_cols].reset_index(drop=True)
        smiles_values = chunk[smiles_col].fillna("").astype(str).tolist()
        desc_rows, graph_rows, fg_rows, fp_rows = [], [], [], []
        for smiles in smiles_values:
            mol = None
            if smiles.strip():
                try:
                    mol = Chem.MolFromSmiles(smiles)
                except Exception:
                    mol = None
            if mol is None:
                total_failed += 1
                desc_rows.append({k: np.nan for k in DESCRIPTOR_NAMES})
                graph_rows.append({k: np.nan for k in GRAPH_NAMES})
                fg_rows.append({k: np.nan for k in FG_NAMES})
                fp_rows.append({f"fp_{j}": np.nan for j in range(int(morgan_nbits))})
            else:
                total_success += 1
                desc_rows.append(_descriptor_values(mol))
                graph_rows.append(_graph_values(mol))
                fg_rows.append({name: float(mol.HasSubstructMatch(q)) for name, q in _COMPILED_FG})
                fp_rows.append(_fingerprint_values(mol, generator, int(morgan_nbits)))

        feature_frames = {
            "descriptors": pd.DataFrame(desc_rows, columns=DESCRIPTOR_NAMES),
            "graph": pd.DataFrame(graph_rows, columns=GRAPH_NAMES),
            "funcgroups": pd.DataFrame(fg_rows, columns=FG_NAMES),
            "fingerprints": pd.DataFrame(fp_rows, columns=[f"fp_{j}" for j in range(int(morgan_nbits))]),
        }
        for ft, feature_frame in feature_frames.items():
            out = pd.concat([base, feature_frame], axis=1)
            out.to_csv(output_paths[ft], mode="a", header=not output_paths[ft].exists(), index=False)

        total_rows += len(chunk)
        elapsed = max(time.time() - start, 1e-9)
        rate = total_rows / elapsed
        eta_min = max(0, (sum(1 for _ in []) or 0))  # placeholder; exact total needs a full scan
        print(
            f"[{stem}] processed {total_rows:,} rows | "
            f"valid {total_success:,} | invalid {total_failed:,} | "
            f"{rate:.1f} rows/s | elapsed {elapsed/60:.1f} min",
            flush=True,
        )

    for ft, path in output_paths.items():
        print(f"[SAVED] {path} ({path.stat().st_size / (1024**3):.2f} GiB)")
    return {
        "rows_processed": int(total_rows),
        "rows_success": int(total_success),
        "rows_failed": int(total_failed),
    }


def main():
    parser = argparse.ArgumentParser(description="Extract polymer RDKit features.")
    parser.add_argument("--input", required=True)
    parser.add_argument("--output", required=True)
    parser.add_argument("--dataset", default=None)
    parser.add_argument("--morgan-radius", type=int, default=2)
    parser.add_argument("--morgan-nbits", type=int, default=2048)
    parser.add_argument("--chunksize", type=int, default=1000)
    args = parser.parse_args()
    input_path = Path(args.input)
    stem = args.dataset or input_path.stem
    stats = process_csv_file(
        input_path, Path(args.output), stem,
        morgan_radius=args.morgan_radius,
        morgan_nbits=args.morgan_nbits,
        chunksize=args.chunksize,
    )
    print("\nExtraction summary:")
    for key, value in stats.items():
        print(f"  {key}: {value:,}")


if __name__ == "__main__":
    main()

Writing /content/extract_features.py


In [ ]:

# ================================================================
# PREDICT YM FROM A NEW POLYMER SMILES
# Uses your existing saved YM experts and MoE router.
# Does NOT retrain any model.
# ================================================================

import os
import importlib.util
import numpy as np
import pandas as pd
import joblib
import xgboost as xgb

from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator

# ------------------------------------------------
# 1. PATHS
# ------------------------------------------------

PROJECT_DIR = "/content/drive/MyDrive/Colab"

MODEL_DIR = os.path.join(
    PROJECT_DIR, "YM_MOE_results", "models"
)

EXTRACTOR_PATH = "/content/extract_features.py"

EXPERTS = [
    "fingerprints",
    "descriptors",
    "graph",
    "functional_groups",
]

FEATURE_COUNTS = {
    "fingerprints": 2048,
    "descriptors": 16,
    "graph": 17,
    "functional_groups": 33,
}

# ------------------------------------------------
# 2. VERIFY REQUIRED FILES
# ------------------------------------------------

required_files = [
    os.path.join(MODEL_DIR, "YM_MoE_router.joblib"),
    EXTRACTOR_PATH,
]

for expert in EXPERTS:
    required_files.append(
        os.path.join(MODEL_DIR, f"YM_{expert}_expert.json")
    )
    required_files.append(
        os.path.join(MODEL_DIR, f"YM_{expert}_medians.npy")
    )

for path in required_files:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"\nRequired file not found:\n{path}\n"
            "Check the path before running predictions."
        )

# ------------------------------------------------
# 3. LOAD THE SAME FEATURE EXTRACTOR
# ------------------------------------------------

spec = importlib.util.spec_from_file_location(
    "extract_features",
    EXTRACTOR_PATH
)

ext = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ext)

# ------------------------------------------------
# 4. LOAD SAVED EXPERT MODELS AND MEDIANS
# ------------------------------------------------

models = {}
medians = {}

for expert in EXPERTS:

    model = xgb.XGBRegressor()

    model.load_model(
        os.path.join(
            MODEL_DIR,
            f"YM_{expert}_expert.json"
        )
    )

    median_values = np.asarray(
        np.load(
            os.path.join(
                MODEL_DIR,
                f"YM_{expert}_medians.npy"
            )
        ),
        dtype=np.float32
    ).reshape(-1)

    if len(median_values) != FEATURE_COUNTS[expert]:
        raise ValueError(
            f"{expert}: expected {FEATURE_COUNTS[expert]} "
            f"medians, found {len(median_values)}."
        )

    models[expert] = model
    medians[expert] = median_values

# Load the trained MoE router.
router = joblib.load(
    os.path.join(
        MODEL_DIR,
        "YM_MoE_router.joblib"
    )
)

# Match the fingerprint settings used for feature extraction.
fingerprint_generator = (
    rdFingerprintGenerator.GetMorganGenerator(
        radius=2,
        fpSize=2048,
        includeChirality=False
    )
)

print("All four experts loaded.")
print("MoE router loaded.")
print("Feature extractor loaded.")

# ------------------------------------------------
# 5. EXTRACT FEATURES FROM ONE SMILES
# ------------------------------------------------

def extract_new_polymer_features(smiles):

    if not isinstance(smiles, str) or not smiles.strip():
        raise ValueError("Enter a non-empty polymer SMILES string.")

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        raise ValueError(
            "RDKit could not parse this SMILES. "
            "Please check its syntax."
        )

    # Same feature-generation functions used in your pipeline.
    raw_features = {
        "fingerprints": ext._fingerprint_values(
            mol,
            fingerprint_generator,
            2048
        ),

        "descriptors": ext._descriptor_values(mol),

        "graph": ext._graph_values(mol),

        "functional_groups": {
            name: float(mol.HasSubstructMatch(query))
            for name, query in ext._COMPILED_FG
        },
    }

    feature_arrays = {}

    for expert in EXPERTS:

        values = raw_features[expert]

        # Enforce the exact expected feature count.
        if len(values) != FEATURE_COUNTS[expert]:
            raise ValueError(
                f"{expert}: expected {FEATURE_COUNTS[expert]} "
                f"features, got {len(values)}."
            )

        # Preserve the order returned by the verified extractor.
        arr = np.asarray(
            list(values.values()),
            dtype=np.float32
        ).reshape(1, -1)

        # Fingerprints should be complete and require no imputation.
        if expert == "fingerprints":
            if not np.isfinite(arr).all():
                raise ValueError(
                    "Fingerprint features contain non-finite values."
                )

        else:
            # Use the saved medians from the corresponding expert.
            missing = ~np.isfinite(arr[0])

            if missing.any():
                arr[0, missing] = medians[expert][missing]

        if not np.isfinite(arr).all():
            raise ValueError(
                f"{expert}: non-finite values remain after preprocessing."
            )

        feature_arrays[expert] = arr

    return feature_arrays


# ------------------------------------------------
# 6. PREDICT YM USING FOUR EXPERTS + MOE ROUTER
# ------------------------------------------------

def predict_polymer_YM(smiles):

    features = extract_new_polymer_features(smiles)

    # Individual expert predictions in the exact router order.
    expert_predictions = {}

    for expert in EXPERTS:

        prediction = models[expert].predict(
            features[expert]
        )[0]

        expert_predictions[expert] = float(prediction)

    prediction_vector = np.array(
        [[expert_predictions[e] for e in EXPERTS]],
        dtype=np.float64
    )

    # Final MoE prediction.
    predicted_YM = float(
        router.predict(prediction_vector)[0]
    )

    # Sample standard deviation across the four expert predictions.
    disagreement_Di = float(
        np.std(prediction_vector[0], ddof=1)
    )

    result = {
        "SMILES": smiles,
        "Predicted_YM": predicted_YM,
        "Disagreement_Di": disagreement_Di,
        **{
            f"{expert}_YM": expert_predictions[expert]
            for expert in EXPERTS
        }
    }

    print("\n" + "=" * 60)
    print("NEW POLYMER — YM PREDICTION")
    print("=" * 60)

    print("SMILES:", smiles)

    print("\nIndividual expert predictions:")
    for expert in EXPERTS:
        print(
            f"  {expert:20s}: "
            f"{expert_predictions[expert]:.6f}"
        )

    print("\nFinal MoE predicted YM:", f"{predicted_YM:.6f}")
    print("Expert disagreement Di:", f"{disagreement_Di:.6f}")

    print("\nNote: Di measures disagreement between experts;")
    print("it is not a calibrated prediction interval.")

    return result


# ------------------------------------------------
# 7. ENTER YOUR NEW POLYMER SMILES HERE
# ------------------------------------------------

NEW_POLYMER_SMILES = "[*]CCCc1cccc(-c2c(Br)cc(-c3c(Br)cc(-c4cccc(N(C)CCC[*])c4)cc3Br)cc2Br)c1"

result = predict_polymer_YM(NEW_POLYMER_SMILES)

# Save the prediction to Drive.
output_path = os.path.join(
    PROJECT_DIR,
    "YM_MOE_results",
    "predictions",
    "new_polymer_YM_prediction.csv"
)

pd.DataFrame([result]).to_csv(
    output_path,
    index=False
)

print("\nPrediction saved to:")
print(output_path)


All four experts loaded.
MoE router loaded.
Feature extractor loaded.

NEW POLYMER — YM PREDICTION
SMILES: [*]CCCc1cccc(-c2c(Br)cc(-c3c(Br)cc(-c4cccc(N(C)CCC[*])c4)cc3Br)cc2Br)c1

Individual expert predictions:
  fingerprints        : 2168.003662
  descriptors         : 2009.210938
  graph               : 1813.159912
  functional_groups   : 1980.480835

Final MoE predicted YM: 2135.866620
Expert disagreement Di: 145.359251

Note: Di measures disagreement between experts;
it is not a calibrated prediction interval.

Prediction saved to:
/content/drive/MyDrive/Colab/YM_MOE_results/predictions/new_polymer_YM_prediction.csv
